# Step 12 — RAG (Retrieval-Augmented Generation)

🇬🇧 **English** (this notebook)

RAG grounds a response in a document you provide: your text is chunked, embedded, and the most relevant chunks are retrieved and injected into the agent's context automatically. Unlike Steps 09–11, `knowledge_sources` does **not** work with a standalone `Agent.kickoff()` call — retrieval only gets wired up through a `Crew`'s own orchestration. So this one exercise needs a minimal inline `Crew` (still defined right here, no external YAML files) instead of a bare `Agent`.

## Learning objective

By the end of this notebook, you will:

- Understand what RAG (Retrieval-Augmented Generation) means, and what kind of information it's for that web search and MCP aren't
- Have pointed a `TextFileKnowledgeSource` at your own document and wired it into a `Crew`
- Be able to demonstrate, by removing the knowledge source, exactly what retrieval added to the answer

## Prerequisites

- [Step 10 — Tools](step_10_tools.ipynb) and [Step 11 — MCP](step_11_mcp.ipynb) completed — this notebook directly compares against both
- A `GEMINI_API_KEY` in your `.env`, even if your chat model (`MODEL`) is a different provider — embeddings are configured to use Gemini explicitly here
- The same `.env` setup as the previous steps otherwise

## Background

Retrieval-Augmented Generation grounds a model's answer in a specific document you provide, instead of relying on what it memorized during training or what a live web search happens to surface:

> Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)

## How this works

Add a `.txt` file with background on your topic to `knowledge/`, then point `TextFileKnowledgeSource` at it:

1. `TextFileKnowledgeSource(file_paths=["ai_act_internal_review.txt"])` tells CrewAI which document to chunk and embed. `knowledge/` is resolved relative to wherever the notebook's kernel is running (this repo's `exercises/en/knowledge/`, by default) — not the repo root.
2. The agent's identity is once again the same Researcher template.
3. `Crew(..., knowledge_sources=[knowledge_source], embedder={...})` is the minimum needed to activate retrieval — this is why RAG needs a `Crew` even for a single agent, unlike Steps 09–11.
4. The question asks about something described only in a private internal document (`knowledge/ai_act_internal_review.txt`, included in this repo) — something no web search or URL fetch could ever find, because it was never published. This is the case RAG is actually for.
5. `tracing=True` — introduced in [Step 02](step_02_intro_to_crewai.ipynb) — prints a shareable dashboard link once the cell finishes; its timeline is a quick way to see the retrieval step actually happening, not just the final grounded answer.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

load_dotenv()

# ── Your knowledge document — a private document, not published anywhere ─────
knowledge_source = TextFileKnowledgeSource(file_paths=["ai_act_internal_review.txt"])

topic = "EU AI Act"

# ── Agent — same "researcher" template as agents.yaml, {topic} filled in via an f-string ─
role      = f"{topic} Senior Data Researcher"
goal      = f"Uncover cutting-edge developments in {topic}"
backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)
agent = Agent(role=role, goal=goal, backstory=backstory, verbose=True)

# ── Task ───────────────────────────────────────────────────────────────────────
question = (
    "According to our internal review notes, is our Resume Triage Assistant "
    f"classified as high-risk under {topic}, and what compliance gaps remain?"
)
task = Task(
    description=question,
    expected_output="A direct answer grounded in the internal review notes.",
    agent=agent,
)

# ── Crew ─────────────────────────────────────────────────────────────────────
crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    knowledge_sources=[knowledge_source],
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes.
    tracing=True,
    embedder={
        "provider": "google-generativeai",
        "config": {"api_key": os.getenv("GEMINI_API_KEY"), "model_name": "gemini-embedding-001"},
    },
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print(result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: According to our internal review notes, is our Resume Triage Assistant classified as high-risk under EU  │
│  AI Act, and what compliance gaps remain?                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To provide you with the exact classification and gap analysis, I have retrieved the **Internal Review Notes:   │
│  Project Resume Triage Assistant (RTA)** from our secure compliance repository.                                 │
│                                                                                                                 │
│  Below is the complete content of the review notes:                                                             │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### **INTERNAL REVIEW: AI ACT CLASSIFICATION & COMPLIANCE GAP ANALYSIS**                                       │
│  **System Name:** Resume Triage Assistant (RTA)                                                                 │
│  **Date of Review:** October 24, 2024                                                                           │
│  **Reviewer:** Regulatory Compliance Lead / AI Act Task Force                                                   │
│                                                                                                                 │
│  #### **1. Classification Status**                                                                              │
│  **Classification:** **High-Risk**                                                                              │
│  **Legal Basis:** Annex III, Point 1(a) of the EU AI Act.                                                       │
│  **Rationale:** The RTA is classified as high-risk because it is intended to be used for the recruitment or     │
│  selection of natural persons, specifically for placing persons in professional internship positions and        │
│  employment. AI systems intended to be used for recruitment or selection, in particular for placing persons in  │
│  professional internship positions, for monitoring and evaluating performance and behavior of persons in such   │
│  relationships, or for making decisions on promotion and termination and for task allocation, fall under the    │
│  scope of Annex III.                                                                                            │
│                                                                                                                 │
│  #### **2. Compliance Gaps Identified**                                                                         │
│  Based on the current audit against Article 9 (Risk Management System) and Article 10 (Data and Data            │
│  Governance), the following gaps remain:                                                                        │
│                                                                                                                 │
│  *   **Gap 1: Data Governance (Article 10)**                                                                    │
│      *   *Finding:* Training, validation, and testing data sets lack comprehensive documentation regarding      │
│  bias mitigation. Specifically, the current dataset includes historical hiring decisions that reflect legacy    │
│  gender imbalances in technical roles.                 

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: e881d44f-b193-4244-9e76-e16b54fa920a                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/e881d44f-b193-4244-9e76-e16b54fa920a?access_code=TRA │
│ CE-175690c771                                                                                                   │
│ 🔑 Access Code: TRACE-175690c771                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

To provide you with the exact classification and gap analysis, I have retrieved the **Internal Review Notes: Project Resume Triage Assistant (RTA)** from our secure compliance repository. 

Below is the complete content of the review notes:

***

### **INTERNAL REVIEW: AI ACT CLASSIFICATION & COMPLIANCE GAP ANALYSIS**
**System Name:** Resume Triage Assistant (RTA)
**Date of Review:** October 24, 2024
**Reviewer:** Regulatory Compliance Lead / AI Act Task Force

#### **1. Classification Status**
**Classification:** **High-Risk**
**Legal Basis:** Annex III, Point 1(a) of the EU AI Act.
**Rationale:** The RTA is classified as high-risk because it is intended to be used for the recruitment or selection of natural persons, specifically for placing persons in professional internship positions and employment. AI systems intended to be used for recruitment or selection, in particular for placing persons in professional internship positions, for monitoring and evaluating performance and behavio

## Your task

1. Run the cell. Does the agent correctly identify the compliance gaps — information that only exists in `knowledge/ai_act_internal_review.txt`, nowhere on the public web?

2. Compare to Steps 10/11: could a web search or a URL fetch ever have answered this question? What kind of information is RAG for that tools/MCP aren't?

3. Now pass an empty list instead (`knowledge_sources=[]`) and rerun. Does the agent still answer correctly, or does it admit it doesn't know? That comparison is the point of RAG.

4. Add your own `.txt` file to `knowledge/` with background on your team's topic, and swap in your own agent identity and question.

5. Note what you observed — it's evidence for `REPORT.md`'s Section 4.2 (Memory) and Section 4.4 (Context Engineering: Context Retrieval). This closes out Sprint 4 — add its row to the **Sprint Progression** table too.

## Shortcomings

Retrieval only surfaces what's already written down in the document you provided — it can't reason across documents it wasn't given, and a poorly written or outdated source document produces a confidently-grounded *wrong* answer just as easily as a correct one. Chunking also loses some context: a fact split across two chunk boundaries may never be retrieved together.

Everything from Step 09 through here has also been **one agent** doing everything itself — the same Researcher that reasons, searches, fetches, and now retrieves is also the one who'd have to turn all of that into a report for a specific audience. [Step 13](step_13_multi_agent.ipynb) closes that gap: splitting research and reporting across two complementary agents.

## Resources for further reading

- Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)
- [CrewAI Knowledge concept docs](https://docs.crewai.com/en/concepts/knowledge)

## Stretch goal

Add a PDF instead of a plain text file:
```python
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
PDFKnowledgeSource(file_paths=["your_document.pdf"])
```
Ask a question whose answer appears on a specific page. Does the agent retrieve it?